# Cell 1: Capture CPU environment
Records machine identity before any campaign. Output: environment text shown below the cell.

In [ ]:
!uname -a
!lscpu
!free -h
!g++ --version
!clang++ --version || true

# Cell 2: Unpack source bundle
Finds the manually uploaded ZIP in Kaggle input. Output: /kaggle/working/SIGMOD source tree.

In [ ]:
from pathlib import Path
import zipfile
bundles = list(Path('/kaggle/input').rglob('kaggle_upload.zip'))
assert len(bundles) == 1, 'Upload the single kaggle_upload.zip dataset first'
root = Path('/kaggle/working/SIGMOD')
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(root)
print(root)

# Cell 3: Inspect source and configs
Output: file listing. Confirm the expected directories before building.

In [ ]:
print('
'.join(str(p.relative_to(root)) for p in sorted(root.rglob('*.json'))))
print((root/'benchmarks/benchmark_unified.cpp').exists())

# Cell 4: Build
Manual execution compiles the C++11 benchmark and test. Output: build/prepared binaries and logs.

In [ ]:
!bash {root}/scripts/linux/build_release.sh --execute

# Cell 5: Small correctness validation
Output: test process status. Stop on failure.

In [ ]:
!{root}/build/prepared/test_v2

# Cell 6: Static smoke subset
Output: one measured CSV and one command log. Inspect correctness and fingerprint first.

In [ ]:
smoke = root/'results/raw/kaggle/smoke.csv'
smoke.parent.mkdir(parents=True, exist_ok=True)
if not smoke.exists():
    import subprocess
    subprocess.run([str(root/'build/prepared/bench_unified'),'--run-id','kaggle-smoke','--output',str(smoke),'--variant','STATIC','--records','1000','--operations','10000','--seed','11'], check=True)
print(smoke)

# Cell 7: Full static opportunity sweep (expensive)
Each completed run checkpoints immediately. Output: results/raw/kaggle/<machine>/static_opportunity/*.csv.

In [ ]:
!bash {root}/scripts/linux/run_static_sweep.sh --execute kaggle

# Cell 8: Oracle processing
Output: measured static fanout envelope CSV. No predicted timings.

In [ ]:
!python3 {root}/analysis/oracle.py --raw {root}/results/raw/kaggle --output {root}/results/processed/oracle.csv

# Cell 9: Dynamic static/regional sweep and regime selection
Run the expensive dynamic_broad sweep only after reviewing the grid and available CPU time. Selection requires five paired seeds and never uses adaptive measurements.

In [ ]:
# Uncomment each command only after manually checking the prior output.
# !bash {root}/scripts/linux/run_campaign.sh experiments/configs/dynamic/broad.json --execute kaggle
# !python3 {root}/analysis/phase_oracle.py --raw {root}/results/raw/kaggle --output {root}/results/processed/phase_oracle.csv
# !python3 {root}/experiments/select_regimes.py --family dynamic_broad --raw {root}/results/raw/kaggle --output {root}/results/manifests/selected_regimes.json

# Cell 10: Dynamic chunk 1
Edit selected_template.json using real selection first. Expensive. Output: checkpointed V1/V2 CSVs.

In [ ]:
# Review and replace RESULT_PENDING in selected_template.json before running.
# !bash {root}/scripts/linux/run_dynamic_selected.sh --execute kaggle

# Cell 11: Dynamic chunk 2
Reserve for a second selected regime with a separate config; do not repeat chunk 1 blindly.

In [ ]:
# !bash {root}/scripts/linux/run_campaign.sh experiments/configs/dynamic/selected_second.json --execute kaggle

# Cell 12: 5M-key scale (expensive; verify RAM)
Output: checkpointed scale campaign. Stop if machine is swapping.

In [ ]:
# !bash {root}/scripts/linux/run_scale.sh --execute kaggle

# Cell 13: Cloud replication subset
Use a reduced config on another Kaggle session. Record CPU model; output stays in separate machine group.

In [ ]:
# !bash {root}/scripts/linux/run_replication.sh --execute kaggle

# Cell 14: Aggregate measured results
Output: results/processed/summary.csv. Warnings flag noisy or small groups.

In [ ]:
!python3 {root}/analysis/aggregate.py --raw {root}/results/raw/kaggle --output {root}/results/processed/summary.csv

# Cell 15: Generate figures/tables
Requires measured oracle and summary CSVs. Output: PDF and PNG in results/figures.

In [ ]:
!python3 {root}/analysis/plot_maps.py --oracle {root}/results/processed/oracle.csv --summary {root}/results/processed/summary.csv --output-dir {root}/results/figures

# Cell 16: Package results for manual download
Output: /kaggle/working/results_download.zip. This cell does not upload anywhere.

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/results_download', 'zip', root/'results')
print(archive)